# Fire Weather Monitoring Gaps on Tribal Lands

**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** RAWS station network (SynopticData / NOAA ISD), Census TIGER AIANNH

---

## Overview

Fire weather monitoring — the network of Remote Automated Weather Stations (RAWS)
that measure temperature, humidity, wind speed, and precipitation — is the
foundation of fire danger rating, prescribed burn decisions, and firefighter
safety. Without nearby stations, fire managers operate with incomplete information.

This notebook quantifies RAWS monitoring coverage relative to each Tribal land
in the study area, identifies monitoring dead zones, and provides a coverage gap
score that directly supports arguments for federal investment in Tribal fire
weather infrastructure.

## Key Questions
- How far is each Tribal land from the nearest active RAWS station?
- Which Tribal lands fall in monitoring dead zones (no station within 50 km)?
- How does RAWS density on and near Tribal lands compare to the national
  rural baseline?
- Are any stations Tribal or BIA-operated, indicating existing Tribal
  investment in fire weather infrastructure?

## RAWS Coverage Standards

| Distance to nearest RAWS | Coverage assessment |
|---|---|
| < 20 km | Adequate — station likely representative of local conditions |
| 20–50 km | Marginal — conditions may differ, especially in complex terrain |
| 50–100 km | Poor — fire weather data may not reflect local conditions |
| > 100 km | Dead zone — no meaningful local fire weather data |

## Data Source

**Primary:** SynopticData API (free token from synopticdata.com) — complete
RAWS network including all federal and Tribal-operated stations.  
**Fallback:** NOAA ISD station history CSV — no token required, filters
for stations with 'RAWS' in name. Less complete, especially for BIA stations.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.validation import make_valid

from src.data import constants, loaders, validators
from src.data.constants import PRIMARY_TRIBES
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import generate_citations, print_data_acknowledgment
from src.viz import charts, styles

styles.apply_mpl_style()
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Output dir: {constants.OUTPUTS_DIR}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
print_data_acknowledgment(source_keys=["census_aiannh", "raws_stations"])

---
## 1. Configuration

In [ ]:
# ── Analysis parameters ────────────────────────────────────────────────────────
TRIBES_OF_INTEREST = PRIMARY_TRIBES

# States covering PRIMARY_TRIBES
STATES = ["AZ", "OK", "WA", "OR"]

# Coverage buffer distances (km)
BUFFER_DISTANCES_KM = [25, 50, 100]

# Coverage gap thresholds (km to nearest station)
ADEQUATE_KM   = 20
MARGINAL_KM   = 50
POOR_KM       = 100
# > POOR_KM = dead zone

# National rural RAWS density benchmark (stations per 1000 km²)
# Based on approximate CONUS RAWS count (~1,800) over rural land area
NATIONAL_RURAL_DENSITY = 0.8

# SynopticData API token (optional — fallback to NOAA ISD if not set)
SYNOPTIC_TOKEN = os.environ.get("SYNOPTIC_TOKEN", None)

print("RAWS MONITORING GAP ANALYSIS")
print("=" * 50)
print(f"  Tribal Nations : {len(TRIBES_OF_INTEREST)}")
print(f"  States         : {', '.join(STATES)}")
print(f"  Buffer zones   : {BUFFER_DISTANCES_KM} km")
print(f"\nData source:")
if SYNOPTIC_TOKEN:
    print("  SynopticData API (token set) — complete RAWS registry")
else:
    print(
        "  NOAA ISD fallback (no SYNOPTIC_TOKEN set)\n"
        "  For complete coverage including BIA stations, register at:\n"
        "  https://synopticdata.com/mesonet-api/ and set SYNOPTIC_TOKEN in .env"
    )

---
## 2. Load Tribal Land Boundaries

In [ ]:
# ── Census TIGER AIANNH ────────────────────────────────────────────────────────
all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh", required_columns=["geometry", "NAME"]
)
tribal_lands = all_tribal[all_tribal["NAME"].isin(TRIBES_OF_INTEREST)].copy()
tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False).reset_index(drop=True)
tribal_lands["geometry"] = tribal_lands.geometry.apply(
    lambda g: make_valid(g) if g is not None else g
)
CONUS = geo_utils.bbox_geodataframe((-127, 24, -65, 50)).geometry.iloc[0]
tribal_lands = tribal_lands[
    tribal_lands.geometry.notnull() &
    tribal_lands.geometry.is_valid &
    tribal_lands.intersects(CONUS)
].copy().reset_index(drop=True)

# Projected centroids for distance calculations
tribal_proj    = tribal_lands.to_crs("EPSG:5070")
centroids_proj = tribal_proj.geometry.centroid
centroids_geo  = centroids_proj.to_crs(constants.CRS_GEOGRAPHIC)
tribal_lands["centroid_lon"] = centroids_geo.x
tribal_lands["centroid_lat"] = centroids_geo.y
tribal_lands["area_km2"]     = tribal_proj.geometry.area / 1e6

print(f"Tribal Nations loaded: {len(tribal_lands)}")
tribal_lands[["NAME", "area_km2", "centroid_lon", "centroid_lat"]].round(2)

---
## 3. Load RAWS Station Locations

In [ ]:
# ── RAWS stations ─────────────────────────────────────────────────────────────
raws = loaders.load_raws_stations(
    states=STATES,
    synoptic_token=SYNOPTIC_TOKEN,
)
raws = validators.validate_geodataframe(
    raws, "raws_stations", required_columns=["geometry", "name"]
)

# Flag Tribal/BIA-operated stations
BIA_KEYWORDS = ["BIA", "TRIBAL", "TRIBE", "NATION", "PUEBLO", "BAND"]
raws["tribal_operated"] = raws["name"].str.upper().apply(
    lambda n: any(kw in n for kw in BIA_KEYWORDS)
) | raws["network"].str.upper().str.contains("BIA", na=False)

print(f"RAWS stations loaded: {len(raws):,}")
print(f"Source             : {raws['source'].iloc[0]}")
print(f"Tribal/BIA operated: {raws['tribal_operated'].sum()}")
print(f"\nBy state:")
print(raws["state"].value_counts().to_string())

---
## 4. Coverage Analysis

In [ ]:
# ── Distance to nearest RAWS and station counts per buffer ────────────────────
tribal_proj = tribal_lands.to_crs("EPSG:5070")
raws_proj   = raws.to_crs("EPSG:5070")

coverage_records = []

for _, tribe in tribal_proj.iterrows():
    centroid = tribe.geometry.centroid

    # Distance from centroid to every RAWS station
    dists_km = raws_proj.geometry.distance(centroid) / 1000

    nearest_idx  = dists_km.idxmin()
    nearest_km   = round(dists_km.min(), 1)
    nearest_name = raws.loc[nearest_idx, "name"]
    nearest_net  = raws.loc[nearest_idx, "network"]

    # Count stations within each buffer
    counts = {f"stations_{d}km": int((dists_km <= d).sum())
              for d in BUFFER_DISTANCES_KM}

    # Station density within 100 km (stations per 1000 km²)
    area_100km = np.pi * 100**2  # km²
    density    = round(counts["stations_100km"] / area_100km * 1000, 3)

    # Tribal/BIA stations within 100 km
    tribal_nearby = int(
        ((dists_km <= 100) & raws["tribal_operated"]).sum()
    )

    # Coverage category
    if nearest_km <= ADEQUATE_KM:
        coverage_cat = "Adequate"
    elif nearest_km <= MARGINAL_KM:
        coverage_cat = "Marginal"
    elif nearest_km <= POOR_KM:
        coverage_cat = "Poor"
    else:
        coverage_cat = "Dead Zone"

    coverage_records.append({
        "NAME":              tribe["NAME"],
        "nearest_km":        nearest_km,
        "nearest_station":   nearest_name,
        "nearest_network":   nearest_net,
        "density_per1000km2": density,
        "tribal_nearby":     tribal_nearby,
        "coverage_category": coverage_cat,
        **counts,
    })

coverage_df = pd.DataFrame(coverage_records)

print("RAWS COVERAGE ANALYSIS")
print("=" * 70)
print(
    coverage_df[[
        "NAME", "nearest_km", "coverage_category",
        f"stations_{BUFFER_DISTANCES_KM[1]}km", "density_per1000km2", "tribal_nearby",
    ]]
    .sort_values("nearest_km", ascending=False)
    .to_string(index=False)
)
print(f"\nNational rural benchmark density: {NATIONAL_RURAL_DENSITY} stations/1000 km²")
print(f"Below benchmark: {(coverage_df['density_per1000km2'] < NATIONAL_RURAL_DENSITY).sum()} of {len(coverage_df)} Tribal Nations")

In [ ]:
# ── Coverage gap score ────────────────────────────────────────────────────────
# Composite score 0–100 where higher = worse coverage.
#
# Distance component (0–60):
#   Scales linearly from 0 (≤20 km) to 60 (≥200 km)
# Density component (0–40):
#   Scales inversely — 0 when density ≥ benchmark, 40 when density = 0

def gap_score(nearest_km: float, density: float) -> float:
    dist_score    = min(nearest_km / 200 * 60, 60)
    density_score = max(0, (1 - density / NATIONAL_RURAL_DENSITY)) * 40
    return round(dist_score + density_score, 1)


coverage_df["gap_score"] = coverage_df.apply(
    lambda r: gap_score(r["nearest_km"], r["density_per1000km2"]), axis=1
)
coverage_df["gap_category"] = pd.cut(
    coverage_df["gap_score"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=["Good", "Moderate Gap", "Significant Gap", "Critical Gap"],
)

print("COVERAGE GAP SCORES")
print("=" * 55)
print(
    coverage_df[["NAME", "nearest_km", "density_per1000km2",
                  "gap_score", "gap_category"]]
    .sort_values("gap_score", ascending=False)
    .to_string(index=False)
)
print(f"\nMean gap score: {coverage_df['gap_score'].mean():.1f}")
print(coverage_df["gap_category"].value_counts().to_string())

---
## 5. Visualizations

In [ ]:
# ── Per-Tribe coverage maps ────────────────────────────────────────────────────
BUFFER_COLORS  = {25: "#27AE60", 50: "#F39C12", 100: "#E74C3C"}
COVERAGE_COLORS = {
    "Adequate":  styles.SAGE_GREEN,
    "Marginal":  styles.FIRE_ORANGE,
    "Poor":      styles.EMBER_RED,
    "Dead Zone": "#8B0000",
}

n_tribes = len(tribal_lands)
ncols, nrows = 2, (n_tribes + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 5))
axes = np.array(axes).flatten()

raws_proj_3857 = raws.to_crs(3857)

for i, (_, tribe_row) in enumerate(tribal_lands.iterrows()):
    ax   = axes[i]
    name = tribe_row["NAME"]
    tb   = tribe_row.geometry.bounds
    buf  = 1.2  # degrees view padding

    clip = geo_utils.bbox_geodataframe(
        (tb[0]-buf, tb[1]-buf, tb[2]+buf, tb[3]+buf)
    ).geometry.iloc[0]

    local_raws = raws[raws.within(clip)].copy()

    # Draw buffer rings (largest first so smaller show on top)
    for d_km in sorted(BUFFER_COLORS.keys(), reverse=True):
        local_raws_proj = local_raws.to_crs("EPSG:5070")
        if not local_raws_proj.empty:
            bufs = local_raws_proj.buffer(d_km * 1000).union_all()
            gpd.GeoDataFrame(
                geometry=[bufs], crs="EPSG:5070"
            ).to_crs(3857).plot(
                ax=ax, color=BUFFER_COLORS[d_km], alpha=0.12, edgecolor="none"
            )

    # Tribal boundary
    cover_row = coverage_df[coverage_df["NAME"] == name]
    cat       = cover_row["coverage_category"].iloc[0] if not cover_row.empty else "Unknown"
    edge_color = COVERAGE_COLORS.get(cat, styles.CHARCOAL)
    gpd.GeoDataFrame(
        geometry=[tribe_row.geometry], crs=constants.CRS_GEOGRAPHIC
    ).to_crs(3857).plot(
        ax=ax, facecolor=edge_color, edgecolor=styles.CHARCOAL,
        alpha=0.25, linewidth=2,
    )

    # RAWS stations
    if not local_raws.empty:
        local_raws_3857 = local_raws.to_crs(3857)
        # Federal stations
        fed = local_raws_3857[~local_raws_3857["tribal_operated"]]
        tri = local_raws_3857[local_raws_3857["tribal_operated"]]
        if not fed.empty:
            fed.plot(ax=ax, color=styles.SKY_BLUE, marker="^",
                     markersize=50, edgecolor="black", linewidth=0.4, zorder=5)
        if not tri.empty:
            tri.plot(ax=ax, color=styles.EMBER_RED, marker="*",
                     markersize=100, edgecolor="black", linewidth=0.4, zorder=6)

    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    except Exception:
        pass

    nearest_km = cover_row["nearest_km"].iloc[0] if not cover_row.empty else "?"
    ax.set_title(
        f"{name}\nNearest RAWS: {nearest_km} km — {cat}",
        fontsize=8, fontweight="bold",
    )
    ax.set_axis_off()

for ax in axes[n_tribes:]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(color=c, alpha=0.3, label=f"{d} km buffer")
        for d, c in sorted(BUFFER_COLORS.items())
    ] + [
        plt.Line2D([0],[0], marker="^", color="w", markerfacecolor=styles.SKY_BLUE,
                   markersize=8, label="Federal RAWS"),
        plt.Line2D([0],[0], marker="*", color="w", markerfacecolor=styles.EMBER_RED,
                   markersize=10, label="Tribal/BIA RAWS"),
    ] + [
        mpatches.Patch(color=v, alpha=0.4, label=k)
        for k, v in COVERAGE_COLORS.items()
    ],
    loc="lower center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, 0),
)
plt.suptitle(
    "RAWS Fire Weather Monitoring Coverage by Tribal Land",
    fontsize=13, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.1, 1, 1])
charts.save_figure(fig, "outputs/figures/raws_coverage_maps.png")
plt.show()

In [ ]:
# ── Coverage gap summary charts ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Chart 1: Distance to nearest RAWS
s = coverage_df.sort_values("nearest_km", ascending=False)
bar_colors = [COVERAGE_COLORS.get(c, styles.SMOKE_GRAY) for c in s["coverage_category"]]
axes[0].barh(s["NAME"], s["nearest_km"], color=bar_colors, alpha=0.85)
for threshold, label, color in [
    (ADEQUATE_KM, f"Adequate ({ADEQUATE_KM} km)", styles.SAGE_GREEN),
    (MARGINAL_KM, f"Marginal ({MARGINAL_KM} km)", styles.FIRE_ORANGE),
    (POOR_KM,     f"Poor ({POOR_KM} km)",          styles.EMBER_RED),
]:
    axes[0].axvline(threshold, color=color, linestyle="--", alpha=0.7, linewidth=1.2, label=label)
axes[0].set_xlabel("Distance to nearest RAWS (km)")
axes[0].set_title("Nearest RAWS Station", fontweight="bold")
axes[0].legend(fontsize=7)
sns.despine(ax=axes[0])

# Chart 2: Station density vs. national benchmark
s2 = coverage_df.sort_values("density_per1000km2", ascending=True)
axes[1].barh(
    s2["NAME"], s2["density_per1000km2"],
    color=[styles.SAGE_GREEN if v >= NATIONAL_RURAL_DENSITY else styles.EMBER_RED
           for v in s2["density_per1000km2"]],
    alpha=0.85,
)
axes[1].axvline(
    NATIONAL_RURAL_DENSITY, color=styles.CHARCOAL,
    linestyle="--", linewidth=1.5,
    label=f"National rural benchmark ({NATIONAL_RURAL_DENSITY})",
)
axes[1].set_xlabel("RAWS stations per 1000 km² (within 100 km)")
axes[1].set_title("Station Density vs. Benchmark", fontweight="bold")
axes[1].legend(fontsize=7)
sns.despine(ax=axes[1])

# Chart 3: Coverage gap score
GAP_COLORS = {
    "Good":             styles.SAGE_GREEN,
    "Moderate Gap":     styles.SKY_BLUE,
    "Significant Gap":  styles.FIRE_ORANGE,
    "Critical Gap":     styles.EMBER_RED,
}
s3 = coverage_df.sort_values("gap_score", ascending=True)
axes[2].barh(
    s3["NAME"], s3["gap_score"],
    color=[GAP_COLORS.get(str(c), styles.SMOKE_GRAY) for c in s3["gap_category"]],
    alpha=0.85,
)
axes[2].set_xlabel("Coverage Gap Score (0–100)")
axes[2].set_title("Coverage Gap Score\n(distance + density)", fontweight="bold")
axes[2].set_xlim(0, 100)
axes[2].legend(
    handles=[mpatches.Patch(color=v, alpha=0.85, label=k) for k, v in GAP_COLORS.items()],
    fontsize=7,
)
sns.despine(ax=axes[2])

plt.suptitle(
    "RAWS Fire Weather Monitoring Gaps — PRIMARY_TRIBES",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/raws_coverage_gap_summary.png")
plt.show()

In [ ]:
# ── Regional overview map: all Tribes + all RAWS stations ─────────────────────
fig, ax = plt.subplots(figsize=(14, 9))

# All RAWS stations
raws_3857 = raws.to_crs(3857)
raws_3857[~raws_3857["tribal_operated"]].plot(
    ax=ax, color=styles.SKY_BLUE, marker="^",
    markersize=15, alpha=0.5, edgecolor="none",
)
raws_3857[raws_3857["tribal_operated"]].plot(
    ax=ax, color=styles.EMBER_RED, marker="*",
    markersize=50, alpha=0.9, edgecolor="black", linewidth=0.3,
)

# Tribal lands colored by gap category
tribal_with_gap = tribal_lands.merge(coverage_df[["NAME", "gap_category", "nearest_km"]], on="NAME")
for cat, color in GAP_COLORS.items():
    sub = tribal_with_gap[tribal_with_gap["gap_category"] == cat]
    if not sub.empty:
        sub.to_crs(3857).plot(
            ax=ax, color=color, alpha=0.5,
            edgecolor=styles.CHARCOAL, linewidth=1.5,
        )

# Labels
for _, row in tribal_with_gap.iterrows():
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    cx3, cy3 = (
        gpd.GeoDataFrame(geometry=[row.geometry.centroid], crs=constants.CRS_GEOGRAPHIC)
        .to_crs(3857).geometry.iloc[0].coords[0]
    )
    ax.annotate(
        f"{row['NAME'].split()[0]}\n{row['nearest_km']} km",
        (cx3, cy3), ha="center", fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7),
    )

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass

ax.set_axis_off()
ax.set_title(
    "RAWS Station Network and Tribal Land Coverage Gaps",
    fontsize=12, fontweight="bold",
)
ax.legend(
    handles=[
        plt.Line2D([0],[0], marker="^", color="w",
                   markerfacecolor=styles.SKY_BLUE, markersize=8, label="Federal RAWS"),
        plt.Line2D([0],[0], marker="*", color="w",
                   markerfacecolor=styles.EMBER_RED, markersize=10, label="Tribal/BIA RAWS"),
    ] + [
        mpatches.Patch(color=v, alpha=0.5, label=k) for k, v in GAP_COLORS.items()
    ],
    loc="lower left", fontsize=8,
)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/raws_regional_overview.png")
plt.show()

---
## 6. Export

In [ ]:
coverage_df.assign(
    gap_category=lambda df: df["gap_category"].astype(str)
).to_csv(
    constants.OUTPUTS_DIR / "raws_coverage_gap_analysis.csv", index=False
)
print("Exported → outputs/raws_coverage_gap_analysis.csv")

raws.to_file(
    constants.OUTPUTS_DIR / "raws_stations.geojson", driver="GeoJSON"
)
print("Exported → outputs/raws_stations.geojson")

tribal_lands.merge(
    coverage_df.drop(columns=["nearest_station", "nearest_network"], errors="ignore"),
    on="NAME",
).assign(
    gap_category=lambda df: df["gap_category"].astype(str)
).to_file(
    constants.OUTPUTS_DIR / "tribal_raws_coverage.geojson", driver="GeoJSON"
)
print("Exported → outputs/tribal_raws_coverage.geojson")

---
## 7. Summary & Findings

*(Fill in after running with real data.)*

**What the data shows:**
- Which Tribal Nations fall in dead zones (nearest RAWS > 100 km)?
- How many Tribal Nations are below the national rural density benchmark?
  This is the core equity argument.
- Are any Tribal/BIA-operated stations present? Their presence indicates
  existing Tribal investment that could be expanded with federal support.

**Connection to the rest of the series:**
- Cross-reference with `fast_fire_days_analysis.ipynb` — Tribes with the
  most fast-fire days AND the worst RAWS coverage face the greatest fire
  management information deficit. Name those Tribes explicitly.
- Cross-reference with `climate_projections_tribal_fire_weather.ipynb` —
  projected increases in fire weather months with inadequate monitoring
  is a compounding risk.

**Policy arguments this analysis supports:**
- BIA Division of Fire Management budget requests for Tribal station installation
- EPA Section 309 / Tribal air quality monitoring funding
- Tribal Climate Resilience Program applications
- Good Neighbor Authority agreements that include monitoring components
- Arguments for NIFC priority listing of Tribal monitoring gaps

**Limitations:**
- Distance from centroid to nearest station uses straight-line distance.
  In complex terrain, topographic barriers mean the effective representativeness
  of a station degrades faster than distance alone implies.
- NOAA ISD fallback undercounts RAWS (misses stations without 'RAWS' in name,
  particularly BIA-operated stations). SynopticData gives a more complete picture.
- Station placement criteria (elevation, aspect, vegetation) matter for
  representativeness — this analysis counts stations, not quality.

---
## References

In [ ]:
print(generate_citations(["census_aiannh", "raws_stations"]))